In [1]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

## ---------- SUBSCRIBERS ----------

In [3]:
subscribers=pd.read_csv(r"C:\Users\daran\Downloads\ff5a102e4361911d42cef9e21cda23d0\subscribers.csv")
print(subscribers)

      subscriber_id signup_date         country         region  age  gender  \
0         SUB100000  2023-01-09          France         Europe   22  Female   
1         SUB100001  2020-02-05  United Kingdom         Europe   48    Male   
2         SUB100002  2022-04-15           Italy         Europe   35    Male   
3         SUB100003  2024-10-10           Spain         Europe   49  Female   
4         SUB100004  2020-04-12   United States  North America   36  Female   
...             ...         ...             ...            ...  ...     ...   
14995     SUB114995  2023-06-14           Italy         Europe   40    Male   
14996     SUB114996  2025-11-29           Japan      East Asia   54  Female   
14997     SUB114997  2025-09-16   United States  North America   22    Male   
14998     SUB114998  2023-06-04           India     South Asia   19    Male   
14999     SUB114999  2023-08-06           Italy         Europe   16    Male   

            plan_type  monthly_price_usd  household

In [4]:
subscribers.shape
subscribers.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   subscriber_id      15000 non-null  str    
 1   signup_date        15000 non-null  str    
 2   country            15000 non-null  str    
 3   region             15000 non-null  str    
 4   age                15000 non-null  int64  
 5   gender             15000 non-null  str    
 6   plan_type          15000 non-null  str    
 7   monthly_price_usd  15000 non-null  float64
 8   household_size     15000 non-null  int64  
 9   primary_device     15000 non-null  str    
 10  payment_method     15000 non-null  str    
 11  tenure_months      15000 non-null  int64  
 12  is_active          15000 non-null  bool   
 13  churn_date         3801 non-null   str    
dtypes: bool(1), float64(1), int64(3), str(9)
memory usage: 1.5 MB


In [5]:
subscribers.isnull().sum()


subscriber_id            0
signup_date              0
country                  0
region                   0
age                      0
gender                   0
plan_type                0
monthly_price_usd        0
household_size           0
primary_device           0
payment_method           0
tenure_months            0
is_active                0
churn_date           11199
dtype: int64

In [6]:
subscribers.duplicated().sum()
print(subscribers["subscriber_id"].duplicated().sum())

0


In [7]:
# Missing values with % of rows affected
nulls = subscribers.isna().sum()
null_report = pd.DataFrame({
    "nulls": nulls,
    "percentage_of_rows": (nulls / len(subscribers) * 100).round(1)
})
print(null_report[null_report["nulls"] > 0])              

            nulls  percentage_of_rows
churn_date  11199                74.7


In [8]:
subscribers["signup_date"]=pd.to_datetime(subscribers["signup_date"],format="%Y-%m-%d")
subscribers["churn_date"]=pd.to_datetime(subscribers["churn_date"],format="%Y-%m-%d")

In [9]:
print(subscribers[["signup_date", "churn_date"]].isna().sum())   
print(subscribers.dtypes[["signup_date", "churn_date"]])         

signup_date        0
churn_date     11199
dtype: int64
signup_date    datetime64[us]
churn_date     datetime64[us]
dtype: object


In [10]:
print(subscribers["is_active"].value_counts())

is_active
True     11199
False     3801
Name: count, dtype: int64


In [11]:
#churn before signup, and active subscribers with a churn date

churned = subscribers[~subscribers["is_active"]]
print("Churn <= signup:", (churned["churn_date"] <= churned["signup_date"]).sum())
print("Active with churn_date:", subscribers[subscribers["is_active"]]["churn_date"].notna().sum())

Churn <= signup: 0
Active with churn_date: 0


In [12]:
#churn dates after the data ends
print((subscribers["churn_date"] > "2026-05-31").sum())

50


#adding a flag column and keeping the rows:

In [13]:
subscribers["churn_after_data_end"] = subscribers["churn_date"] > "2026-05-31"

In [14]:
print((subscribers["age"] < 18).sum()) 

1270


In [15]:
print(((subscribers["tenure_months"] <= 1) & subscribers["is_active"]).sum())  

321


In [16]:
print(subscribers.groupby("plan_type")["monthly_price_usd"].unique())   

plan_type
Basic with Ads     [6.99]
Premium           [22.99]
Standard          [15.49]
Name: monthly_price_usd, dtype: object


In [17]:
# each country maps to exactly one region
print((subscribers.groupby("country")["region"].nunique() > 1).sum())          

# tenure_months agrees with signup date (reference date 2026-06-01)
exp = ((pd.Timestamp("2026-06-01") - subscribers["signup_date"]).dt.days / 30.44).round()
print(((exp - subscribers["tenure_months"]).abs() > 2).sum())                 

0
0


In [18]:
# Export the cleaned subscribers table
subscribers.to_csv("subscribers_clean.csv", index=False)

# Verify the export
check = pd.read_csv("subscribers_clean.csv")
print(check.shape)         
print(check.columns.tolist())

(15000, 15)
['subscriber_id', 'signup_date', 'country', 'region', 'age', 'gender', 'plan_type', 'monthly_price_usd', 'household_size', 'primary_device', 'payment_method', 'tenure_months', 'is_active', 'churn_date', 'churn_after_data_end']


Fixed:

# Converted signup_date and churn_date from text to datetime using the explicit format %Y-%m-%d. An earlier attempt with the wrong format (%d-%m-%Y) and errors="coerce" silently turned all 15,000 dates into NaT. It was caught by comparing null counts, and fixed by reloading subscribers.csv before converting. Nulls after the fix: signup_date 0, churn_date 11,199.
# Removed errors="coerce" from the date conversion so a malformed date raises an error instead of being silently blanked.


Validated (all passed clean):

# No duplicate rows, no duplicate subscriber_ids (15,000 unique).
# Row count matches the PRD (15,000 rows, 14 columns).
# No missing values in any column except churn_date, and its 11,199 nulls are legitimate: they equal the number of active subscribers.
# No churned subscriber has a missing churn_date.
# No active subscriber has a churn_date.
# No churn_date on or before signup_date.
# Each plan_type has exactly one price (Basic with Ads 6.99, Standard 15.49, Premium 22.99).
# Each country maps to exactly one region.
# tenure_months is consistent with signup_date (reference date 2026-06-01).
# age (13-85) and household_size (1-4) are within sane ranges.


Added columns:

# churn_after_data_end: True for the 50 subscribers whose churn_date falls after 2026-05-31, the last date in the viewing data. Kept, since they may be scheduled cancellations; the flag lets churn-timing analysis separate them.


Noted (not fixed):

# 1,270 subscribers are under 18 (ages 13-17). The values are valid, but they matter for maturity-rating analysis.
# 321 active subscribers have tenure of 1 month or less. They are plausible new sign-ups, but they will pull down average engagement metrics.
# 50 churn dates after the data end (see flag above) should be treated with caution in churn-timing analysis.


Result: subscribers table (15,000 rows) is fully cleaned and validated. No rows were removed.Exported as subscribers_clean.csv (original 14 columns) and subscribers_clean.csv ( One column was added (churn_after_data_end), giving 15 columns).

## ---------- TITLES ----------

In [19]:
titles = pd.read_csv(r"C:\Users\daran\Downloads\ff5a102e4361911d42cef9e21cda23d0\titles.csv")
print(titles.shape) 
titles.info()

(9000, 21)
<class 'pandas.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   title_id              9000 non-null   str    
 1   title_name            9000 non-null   str    
 2   type                  9000 non-null   str    
 3   primary_genre         9000 non-null   str    
 4   country               9000 non-null   str    
 5   language              9000 non-null   str    
 6   release_year          9000 non-null   int64  
 7   date_added            9000 non-null   str    
 8   maturity_rating       9000 non-null   str    
 9   seasons               9000 non-null   int64  
 10  content_duration_min  9000 non-null   int64  
 11  is_original           9000 non-null   bool   
 12  license_type          9000 non-null   str    
 13  director              9000 non-null   str    
 14  cast                  9000 non-null   str    
 15  quality_score        

In [20]:
print(titles.isnull().sum())
print(titles.duplicated().sum())
print(titles["title_id"].duplicated().sum())


title_id                   0
title_name                 0
type                       0
primary_genre              0
country                    0
language                   0
release_year               0
date_added                 0
maturity_rating            0
seasons                    0
content_duration_min       0
is_original                0
license_type               0
director                   0
cast                       0
quality_score              0
popularity_score           0
license_cost_usd           0
license_expiry          1966
total_watch_hours          0
total_plays                0
dtype: int64
0
0


In [21]:
# Missing values with % of rows affected
nulls = titles.isna().sum()
null_report = pd.DataFrame({
    "nulls": nulls,
    "percentage_of_rows": (nulls / len(titles) * 100).round(1)
})
print(null_report[null_report["nulls"] > 0])             


                nulls  percentage_of_rows
license_expiry   1966                21.8


In [22]:
titles["date_added"]     = pd.to_datetime(titles["date_added"],     format="%Y-%m-%d")
titles["license_expiry"] = pd.to_datetime(titles["license_expiry"], format="%Y-%m-%d")
print(titles[["date_added", "license_expiry"]].isna().sum())  

date_added           0
license_expiry    1966
dtype: int64


In [23]:
#Check that the nulls are legitimate
print(titles["is_original"].sum())                                          
print((titles["is_original"] & titles["license_expiry"].notna()).sum())    
print((~titles["is_original"] & titles["license_expiry"].isna()).sum())     

1966
0
0


#Check consistency between columns


In [24]:
# Movies should have 0 seasons, TV shows at least 1
print(((titles["type"] == "Movie") & (titles["seasons"] > 0)).sum())        
print(((titles["type"] == "TV Show") & (titles["seasons"] == 0)).sum())     


0
0


In [25]:
# is_original should match license_type
print((titles["is_original"] != (titles["license_type"] == "Original")).sum())

0


In [26]:
# Titles added before their release year
print((titles["date_added"].dt.year < titles["release_year"]).sum())  

0


#Check the value ranges and categories

In [27]:

print(titles[["quality_score", "popularity_score", "license_cost_usd",
              "content_duration_min", "release_year"]].describe().round(2))

       quality_score  popularity_score  license_cost_usd  \
count        9000.00           9000.00           9000.00   
mean           62.72              0.66        2357146.91   
std            15.85              2.64        2916385.20   
min             5.00              0.00          35974.00   
25%            51.90              0.06         745279.00   
50%            62.70              0.19        1467892.00   
75%            73.60              0.54        2827106.25   
max            99.00            100.00       48570366.00   

       content_duration_min  release_year  
count               9000.00       9000.00  
mean                 482.04       2011.36  
std                  623.52         16.63  
min                   45.00       1969.00  
25%                  103.00       2000.00  
50%                  128.50       2019.00  
75%                  720.00       2025.00  
max                 5985.00       2026.00  


In [28]:
print(titles["type"].value_counts())
print(titles["license_type"].value_counts())
print(titles["maturity_rating"].value_counts())

type
Movie      5336
TV Show    3664
Name: count, dtype: int64
license_type
Co-Production            2396
Non-Exclusive License    2336
Exclusive License        2302
Original                 1966
Name: count, dtype: int64
maturity_rating
TV-14    2025
TV-MA    1809
PG-13    1234
R        1141
TV-PG     926
PG        713
TV-G      364
G         328
TV-Y      303
NC-17     157
Name: count, dtype: int64


#The popularity_score problem

In [29]:
print((titles["popularity_score"] > 1).sum())
print(titles.loc[titles["popularity_score"] > 1, 
   ["title_id", "title_name", "popularity_score"]].sort_values("popularity_score", ascending=False).head(10))

1285
       title_id                   title_name  popularity_score
5962  TTL205962  The Shadows of Marseille IV            100.00
7141  TTL207141          The Tide of Oslo IV             99.81
3531  TTL203531                Gilded Throne             80.55
6337  TTL206337      The Cipher of Madrid II             70.56
7554  TTL207554         Restless Shadows III             57.33
2855  TTL202855                 Broken Heist             42.11
6938  TTL206938         Hamburg Frequency IV             41.18
4514  TTL204514                 Neon Horizon             40.83
4462  TTL204462              Lagos Cipher II             37.78
5320  TTL205320           Jakarta Odyssey IV             33.17


In [30]:
titles["popularity_above_1"] = titles["popularity_score"] > 1
print(titles["popularity_above_1"].sum())

1285


#Duplicate titles

In [31]:
# Same name, different title_id
dupes = titles[titles["title_name"].duplicated(keep=False)].sort_values("title_name")
print(len(dupes))
print(dupes[["title_id", "title_name", "release_year", "primary_genre"]].head(10))


85
       title_id            title_name  release_year primary_genre
7087  TTL207087     Berlin Affair VII          1969   Documentary
6629  TTL206629     Berlin Affair VII          2016        Horror
3113  TTL203113    Detroit Garden VII          2026        Horror
3308  TTL203308    Detroit Garden VII          2024         Crime
4285  TTL204285    Detroit Garden VII          2025         Drama
8552  TTL208552     Falling Seoul VII          2026        Action
7899  TTL207899     Falling Seoul VII          2015        Action
8110  TTL208110     Falling Seoul VII          2002         Drama
3622  TTL203622  Forgotten Sydney VII          2005         Drama
5470  TTL205470  Forgotten Sydney VII          2025         Crime


#expired licences

In [32]:
# Licences that expired before the data ends
titles["license_expired"] = titles["license_expiry"] < "2026-06-01"
print(titles["license_expired"].sum())

836


In [33]:
titles.to_csv("titles_clean.csv", index=False)
print(pd.read_csv("titles_clean.csv").shape)     

(9000, 23)


Fixed:

# Converted date_added and license_expiry to datetime using the explicit format %Y-%m-%d. Null counts were compared before and after the conversion: date_added 0, license_expiry 1,966 (unchanged).


Validated (all passed clean):

# Row count matches the PRD (9,000 rows, 21 columns). No duplicate rows, no duplicate title_ids.
# The 1,966 null license_expiry values are legitimate: they equal the number of Originals, no Original has an expiry date, and no licensed title is missing one.
# Movies all have 0 seasons and TV shows all have at least 1.
# is_original agrees with license_type == "Original" for every row.
# No title was added to the catalogue before its release year.
# type, license_type and maturity_rating contain only valid categories (no typos or casing issues).
# quality_score (5-99), license_cost_usd and content_duration_min have no negative or zero values.
# content_duration_min of up to 5,985 min is valid: for TV shows it is the total runtime across seasons.


Added columns:

# popularity_above_1: True for 1,285 titles (14.3%) whose popularity_score exceeds the 0-1 range in the data dictionary. Kept, because the values form one continuous right-skewed distribution (median 0.19, 99th percentile 7.12, maximum 100) and correlate 0.997 with total_plays, so they are genuine, not a second scale. Rescaling would distort rankings.
# license_expired: True for 836 titles whose license_expiry is before 2026-06-01, the end of the data period. Kept, for the licence-expiry analysis.


Noted (not fixed):

# The data dictionary describes popularity_score as 0-1, but the data does not follow it. Analysis should use log scale or percentiles.
# 32 title names are shared by 85 different titles (different title_id, year and genre). They are distinct titles, so none were removed. Joins must use title_id, never title_name.


Result: titles table (9,000 rows) is fully cleaned and validated. No rows were removed. Exported as titles_clean.csv with Two flag columns were added, giving 23 columns.

## ---------- WATCH_HISTORY ----------

In [34]:
watch = pd.read_csv(r"C:\Users\daran\Downloads\ff5a102e4361911d42cef9e21cda23d0\watch_history.csv")
print(watch.shape) 
print(watch.info())                                  

(650000, 10)
<class 'pandas.DataFrame'>
RangeIndex: 650000 entries, 0 to 649999
Data columns (total 10 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   watch_id              650000 non-null  int64  
 1   subscriber_id         650000 non-null  str    
 2   title_id              650000 non-null  str    
 3   watch_date            650000 non-null  str    
 4   device                650000 non-null  str    
 5   region                650000 non-null  str    
 6   content_duration_min  650000 non-null  int64  
 7   watch_duration_min    650000 non-null  float64
 8   completion_pct        650000 non-null  float64
 9   completed             650000 non-null  bool   
dtypes: bool(1), float64(2), int64(2), str(5)
memory usage: 45.3 MB
None


In [35]:
print(watch.isna().sum())                     
print(watch["watch_id"].duplicated().sum()) 

watch_id                0
subscriber_id           0
title_id                0
watch_date              0
device                  0
region                  0
content_duration_min    0
watch_duration_min      0
completion_pct          0
completed               0
dtype: int64
0


In [36]:
watch["watch_date"] = pd.to_datetime(watch["watch_date"], format="%Y-%m-%d")
print(watch["watch_date"].isna().sum())     

0


#Referential integrity

In [37]:
print((~watch["subscriber_id"].isin(subscribers["subscriber_id"])).sum())   
print((~watch["title_id"].isin(titles["title_id"])).sum())                  

0
0


#Duration and completion checks

In [38]:
# Sessions longer than the content itself
print((watch["watch_duration_min"] > watch["content_duration_min"]).sum())  

0


In [39]:
# Non-positive durations
print((watch["watch_duration_min"] <= 0).sum()) 

0


In [40]:
# completion_pct recomputed from the durations
calc = (watch["watch_duration_min"] / watch["content_duration_min"] * 100).clip(upper=100)
print(((calc - watch["completion_pct"]).abs() > 5).sum())                    


0


In [41]:
# Range and the completed flag
print(watch["completion_pct"].min(), watch["completion_pct"].max())        
print(watch.groupby("completed")["completion_pct"].agg(["min", "max"]))       

1.0 100.0
            min    max
completed             
False       1.0   90.0
True       90.0  100.0


In [42]:
at90 = watch[watch["completion_pct"] == 90.0]
print(at90["completed"].value_counts())

completed
False    535
True     524
Name: count, dtype: int64


In [43]:
at90 = watch[watch["completion_pct"] == 90.0].copy()
at90["raw"] = at90["watch_duration_min"] / at90["content_duration_min"] * 100
print(at90.groupby("completed")["raw"].agg(["min", "max"]))

                min       max
completed                    
False      89.90566  90.00000
True       90.00000  90.08547


# Check: `completed` flag vs. 90% threshold
`completed` is True from 90% completion, but 1,059 sessions sit at exactly
`completion_pct` = 90.0 and fall in both groups. Recomputing the ratio from the stored
durations separates them cleanly (False up to 90.00%, True from 90.00%), so the overlap
is a rounding artefact. Left unchanged (0.16% of rows).

#Categories and consistency with titles

In [44]:
print(watch["device"].value_counts())    
print(watch["region"].value_counts())  

device
Smart TV           221446
Mobile             175605
Laptop              96408
Tablet              65276
Game Console        52322
Streaming Stick     38943
Name: count, dtype: int64
region
North America     197412
Europe            159935
East Asia          79829
Latin America      77910
South Asia         74521
Middle East        21903
Southeast Asia     14996
Africa             11950
Oceania            11544
Name: count, dtype: int64


In [45]:
# content_duration_min should match the titles table
m = watch.merge(titles[["title_id", "content_duration_min"]], on="title_id", suffixes=("", "_t"))
print((m["content_duration_min"] != m["content_duration_min_t"]).sum())     

0


#session region against the subscriber’s region

In [46]:
m = watch.merge(subscribers[["subscriber_id", "region"]], on="subscriber_id", suffixes=("", "_sub"))
print((m["region"] != m["region_sub"]).sum())
print((m["region"] != m["region_sub"]).mean().round(3))

0
0.0


#sessions after churn

In [47]:
m = watch.merge(subscribers[["subscriber_id", "signup_date", "churn_date"]], on="subscriber_id")

print((m["watch_date"] < m["signup_date"]).sum())      
after = m[m["watch_date"] > m["churn_date"]]
print(len(after), after["subscriber_id"].nunique())      
print((after["watch_date"] - after["churn_date"]).dt.days.describe())

0
131199 3708
count    131199.000000
mean        665.163843
std         650.759112
min           1.000000
25%         136.000000
50%         447.000000
75%        1018.000000
max        3710.000000
dtype: float64


In [48]:
churn_map = subscribers.set_index("subscriber_id")["churn_date"]
watch["after_churn"] = watch["watch_date"] > watch["subscriber_id"].map(churn_map)

print(watch.shape)                  
print(watch["after_churn"].sum())   

(650000, 11)
131199


In [49]:
hours = watch["watch_duration_min"] / 60
print((hours[watch["after_churn"]].sum() / hours.sum()).round(3))   # share of all watch hours
print(watch["after_churn"].mean().round(3))                         # share of sessions

0.202
0.202


In [50]:
watch.to_csv("watch_history_clean.csv", index=False)
print(pd.read_csv("watch_history_clean.csv").shape)    

(650000, 11)


Fixed:

# Converted watch_date from text to datetime using the explicit format %Y-%m-%d. Nulls after conversion: 0.


Validated (all passed clean):

# Row count matches the PRD (650,000 rows, 10 columns). No missing values in any column. No duplicate watch_ids.
# Every subscriber_id and title_id exists in the subscribers and titles tables (0 orphans).
# No session is longer than its content (watch_duration_min never exceeds content_duration_min) and no duration is zero or negative.
# completion_pct matches the value recomputed from the durations to within 5 points on every row, and ranges from 1.0 to 100.0.
# content_duration_min matches the titles table on every row.
# device (6 categories) and region (9 categories) contain no typos or inconsistent names.
# The session region equals the subscriber's region on every row.
# No session is dated before the subscriber's signup_date.


Added columns:

# after_churn: True for the 131,199 sessions (20.2% of sessions and 20.2% of watch hours) dated after the subscriber's churn_date. Kept, because deleting a fifth of the fact table would change every engagement number. The flag allows KPIs to be computed with and without these rows.


Noted (not fixed):

# 131,199 sessions from 3,708 subscribers (about 98% of the 3,801 churned subscribers) are dated after the churn date. The median is 447 days after churn and the maximum is 3,710 days, so this is a systematic inconsistency between watch_date and churn_date, not a few late sessions. The data does not show which column is wrong, so neither was altered. Phase 3 KPIs will be checked with and without these rows.
# The completed flag turns True from 90% completion. 1,059 sessions sit at exactly completion_pct = 90.0 (535 not completed, 524 completed). Recomputing the ratio from the stored durations separates them to within rounding of watch_duration_min, so they were left unchanged (0.16% of rows).


Result: watch_history table (650,000 rows) is fully cleaned and validated. No rows were removed. One column was added (after_churn), giving 11 columns. Exported as watch_history_clean.csv.

## ---------- RATINGS ----------

In [51]:
ratings = pd.read_csv(r"C:\Users\daran\Downloads\ff5a102e4361911d42cef9e21cda23d0\ratings.csv")
print(ratings.shape) 
print(ratings.info()) 

(130000, 5)
<class 'pandas.DataFrame'>
RangeIndex: 130000 entries, 0 to 129999
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   rating_id      130000 non-null  int64
 1   subscriber_id  130000 non-null  str  
 2   title_id       130000 non-null  str  
 3   rating         130000 non-null  int64
 4   rating_date    130000 non-null  str  
dtypes: int64(2), str(3)
memory usage: 5.0 MB
None


In [52]:
print(ratings.isna().sum().sum()) 
print(ratings.duplicated().sum())  
print(ratings["rating_id"].duplicated().sum())  
print(ratings.duplicated(["subscriber_id", "title_id"]).sum())

0
0
0
200


In [53]:
ratings["rating_date"] = pd.to_datetime(ratings["rating_date"], format="%Y-%m-%d")
print(ratings["rating_date"].isna().sum())   

0


In [54]:
print(ratings["rating"].value_counts().sort_index())  

rating
1     1786
2    14570
3    44491
4    48564
5    20589
Name: count, dtype: int64


In [55]:
print((~ratings["subscriber_id"].isin(subscribers["subscriber_id"])).sum())   
print((~ratings["title_id"].isin(titles["title_id"])).sum())          

0
0


In [56]:
k = ["subscriber_id", "title_id"]

d = ratings[ratings.duplicated(k, keep=False)].sort_values(k + ["rating_date"])
print(d.groupby(k)["rating"].nunique().gt(1).sum())   

ratings_clean = (ratings.sort_values(["rating_date", "rating_id"])
                        .drop_duplicates(k, keep="last")
                        .sort_values("rating_id")
                        .reset_index(drop=True))

print(ratings_clean.shape)                               
print(ratings_clean.duplicated(k).sum())                

123
(129800, 5)
0


In [57]:
m = ratings_clean.merge(subscribers[["subscriber_id", "signup_date", "churn_date"]], on="subscriber_id")
print((m["rating_date"] < m["signup_date"]).sum())    
print((m["rating_date"] > m["churn_date"]).sum())    

pairs = watch[["subscriber_id", "title_id"]].drop_duplicates()
chk = ratings_clean.merge(pairs, on=k, how="left", indicator=True)
print((chk["_merge"] == "left_only").sum(), len(ratings_clean))   

25737
21330
127983 129800


In [58]:
ratings_clean = ratings_clean.merge(subscribers[["subscriber_id", "signup_date", "churn_date"]],
                                    on="subscriber_id", how="left")
ratings_clean["rated_before_signup"] = ratings_clean["rating_date"] < ratings_clean["signup_date"]
ratings_clean["rated_after_churn"]   = ratings_clean["rating_date"] > ratings_clean["churn_date"]
ratings_clean = ratings_clean.drop(columns=["signup_date", "churn_date"])

print(ratings_clean.shape)                       
print(ratings_clean[["rated_before_signup", "rated_after_churn"]].sum())   



(129800, 7)
rated_before_signup    25737
rated_after_churn      21330
dtype: int64


In [59]:
ratings_clean.to_csv("ratings_clean.csv", index=False)
print(pd.read_csv("ratings_clean.csv").shape)        

(129800, 7)


Fixed:

# Converted rating_date from text to datetime using the explicit format %Y-%m-%d. Nulls after conversion: 0.
# Removed 200 duplicate ratings, the extra copies of 199 subscriber-title pairs (198 pairs rated twice, 1 pair rated three times). 123 of those pairs had conflicting scores. Kept the most recent rating for each pair (ties on the same date broken by the higher rating_id), on the assumption that a subscriber's latest rating reflects their current opinion. 130,000 rows became 129,800.


Validated (all passed clean):

# Row count matched the PRD (130,000 rows, 5 columns) before cleaning. No missing values. No duplicate rating_ids.
# rating contains only the values 1 to 5 (1: 1,786, 2: 14,570, 3: 44,491, 4: 48,564, 5: 20,589).
# Every subscriber_id and title_id exists in the subscribers and titles tables (0 orphans).
# No duplicate subscriber-title pairs remain after cleaning.


Added columns:

# rated_before_signup: True for 25,737 ratings (19.8%) dated before the subscriber's signup_date. Kept and flagged.
# rated_after_churn: True for 21,330 ratings (16.4%) dated after the subscriber's churn_date. Kept and flagged.


Noted (not fixed):

# 127,983 of the 129,800 ratings (98.6%) are for a title the subscriber has no session for in watch_history. Ratings therefore cannot be linked to viewing behaviour, and Phase 3 will not treat a rating as evidence that the title was watched.
# The rating dates are inconsistent with subscription dates in the same way the viewing dates are (see watch_history: 131,199 sessions after churn). All of these point to date columns that were generated independently of the subscription timeline. No dates were altered.


Result: ratings table (130,000 rows) is cleaned and validated. 200 duplicate rows were removed, leaving 129,800. Two columns were added (rated_before_signup, rated_after_churn), giving 7 columns. Exported as ratings_clean.csv.

## ---------- REVIEWS ----------

In [60]:
reviews = pd.read_csv(r"C:\Users\daran\Downloads\ff5a102e4361911d42cef9e21cda23d0\reviews.csv")
print(reviews.shape)    
print(reviews.info())    


(110000, 7)
<class 'pandas.DataFrame'>
RangeIndex: 110000 entries, 0 to 109999
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   review_id      110000 non-null  int64
 1   subscriber_id  110000 non-null  str  
 2   title_id       110000 non-null  str  
 3   review_text    110000 non-null  str  
 4   sentiment      110000 non-null  str  
 5   helpful_votes  110000 non-null  int64
 6   review_date    110000 non-null  str  
dtypes: int64(2), str(5)
memory usage: 5.9 MB
None


In [61]:
print(reviews.isna().sum())      
print(reviews.duplicated().sum())                
print(reviews["review_id"].duplicated().sum())

review_id        0
subscriber_id    0
title_id         0
review_text      0
sentiment        0
helpful_votes    0
review_date      0
dtype: int64
0
0


In [62]:
reviews["review_date"] = pd.to_datetime(reviews["review_date"], format="%Y-%m-%d")
print(reviews["review_date"].isna().sum()) 

0


# Sentiment values

In [63]:
print(reviews["sentiment"].value_counts(dropna=False))  

sentiment
Positive    75007
Neutral     22759
Negative    12234
Name: count, dtype: int64


# Integrity and ranges

In [64]:
print((~reviews["subscriber_id"].isin(subscribers["subscriber_id"])).sum())   
print((~reviews["title_id"].isin(titles["title_id"])).sum())                  

0
0


In [65]:
print((reviews["helpful_votes"] < 0).sum())
print(reviews["helpful_votes"].min(), reviews["helpful_votes"].max())     

0
0 24


In [66]:
print(reviews["review_text"].nunique())                  

20


## Duplicate subscriber-title pairs

In [67]:
print(reviews.duplicated(["subscriber_id", "title_id"]).sum())  

171


In [68]:
k = ["subscriber_id", "title_id"]
# Conflicting pairs
d = reviews[reviews.duplicated(k, keep=False)]
print(d.groupby(k)["sentiment"].nunique().gt(1).sum())

81


In [69]:
# Keep the most recent review per pair (tie: higher review_id)
reviews_clean = (reviews.sort_values(["review_date", "review_id"])
                        .drop_duplicates(k, keep="last")
                        .sort_values("review_id")
                        .reset_index(drop=True))
print(reviews_clean.shape)                                  
print(reviews_clean.duplicated(k).sum())                   

(109829, 7)
0


In [70]:
m = reviews_clean.merge(subscribers[["subscriber_id", "signup_date", "churn_date"]], on="subscriber_id", how="left")
reviews_clean["reviewed_before_signup"] = (m["review_date"] < m["signup_date"]).values
reviews_clean["reviewed_after_churn"]   = (m["review_date"] > m["churn_date"]).values
print(reviews_clean[["reviewed_before_signup", "reviewed_after_churn"]].sum())

reviewed_before_signup    22062
reviewed_after_churn      17708
dtype: int64


In [71]:
pairs = watch[k].drop_duplicates()
chk = reviews_clean.merge(pairs, on=k, how="left", indicator=True)
print((chk["_merge"] == "left_only").sum(), len(reviews_clean))


108341 109829


In [72]:
print(reviews_clean.shape)
reviews_clean.to_csv("reviews_clean.csv", index=False)
print(pd.read_csv("reviews_clean.csv").shape)

(109829, 9)
(109829, 9)


Fixed:

# Converted review_date from text to datetime using the explicit format %Y-%m-%d. Nulls after conversion: 0.
# Removed 171 duplicate reviews, the extra copies of 171 subscriber-title pairs (each reviewed twice). 81 of those pairs had conflicting sentiment. Kept the most recent review for each pair (ties on the same date broken by the higher review_id), on the assumption that the latest review reflects the subscriber's current opinion. 110,000 rows became 109,829.


Validated (all passed clean):

# Row count matched the PRD (110,000 rows, 7 columns) before cleaning. No missing values. No duplicate review_ids.
# sentiment contains only Positive (75,007), Neutral (22,759) and Negative (12,234). Nothing else.
# Every subscriber_id and title_id exists in the subscribers and titles tables (0 orphans).
# helpful_votes has no negative values (range 0 to 24).
# No duplicate subscriber-title pairs remain after cleaning.


Added columns:

# reviewed_before_signup: True for 22,062 reviews (20.1%) dated before the subscriber's signup_date. Kept and flagged.
# reviewed_after_churn: True for 17,708 reviews (16.1%) dated after the subscriber's churn_date. Kept and flagged.


Noted (not fixed):

# review_text has only 20 distinct values across 110,000 rows, so the text is templated. Text analysis would add nothing; Phase 2 and 3 use the sentiment column instead.
# 108,341 of the 109,829 reviews (98.6%) are for a title the subscriber has no session for in watch_history, so reviews cannot be tied to viewing behaviour.
# Review dates are inconsistent with subscription dates, as with ratings and viewing sessions. No dates were altered.


Result: reviews table (110,000 rows) is cleaned and validated. 171 duplicate rows were removed, leaving 109,829. Two columns were added (reviewed_before_signup, reviewed_after_churn), giving 9 columns. Exported as reviews_clean.csv.

## ---------- WATCHLIST ----------

In [73]:
watchlist = pd.read_csv(r"C:\Users\daran\Downloads\ff5a102e4361911d42cef9e21cda23d0\watchlist.csv")
print(watchlist.shape)                                  
print(watchlist.info())   

(65000, 5)
<class 'pandas.DataFrame'>
RangeIndex: 65000 entries, 0 to 64999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   watchlist_id   65000 non-null  int64
 1   subscriber_id  65000 non-null  str  
 2   title_id       65000 non-null  str  
 3   added_date     65000 non-null  str  
 4   watched        65000 non-null  bool 
dtypes: bool(1), int64(1), str(3)
memory usage: 2.0 MB
None


In [74]:
print(watchlist.isna().sum())     
print(watchlist.duplicated().sum())    
print(watchlist["watchlist_id"].duplicated().sum())      

watchlist_id     0
subscriber_id    0
title_id         0
added_date       0
watched          0
dtype: int64
0
0


In [75]:
watchlist["added_date"] = pd.to_datetime(watchlist["added_date"], format="%Y-%m-%d")
print(watchlist["added_date"].isna().sum())

0


In [76]:
print(watchlist["watched"].value_counts())

watched
False    34952
True     30048
Name: count, dtype: int64


In [77]:
#referential integrity
print((~watchlist["subscriber_id"].isin(subscribers["subscriber_id"])).sum())
print((~watchlist["title_id"].isin(titles["title_id"])).sum())

0
0


In [78]:
# Duplicate subscriber-title pairs
k = ["subscriber_id", "title_id"]
print(watchlist.duplicated(k).sum())  

61


In [79]:
## Conflicting pairs
d = watchlist[watchlist.duplicated(k, keep=False)]
print(d.groupby(k)["watched"].nunique().gt(1).sum())

31


In [80]:
# Keep the most recent entry per pair (tie: higher watchlist_id)
watchlist_clean = (watchlist.sort_values(["added_date", "watchlist_id"])
                            .drop_duplicates(k, keep="last")
                            .sort_values("watchlist_id")
                            .reset_index(drop=True))
print(watchlist_clean.shape)
print(watchlist_clean.duplicated(k).sum())


(64939, 5)
0


In [81]:
# Date consistency
m = watchlist_clean.merge(subscribers[["subscriber_id", "signup_date", "churn_date"]], on="subscriber_id", how="left")
print((m["added_date"] < m["signup_date"]).sum())    
print((m["added_date"] > m["churn_date"]).sum())

0
7774


In [82]:
watchlist_clean["added_after_churn"] = (m["added_date"] > m["churn_date"]).values
print(watchlist_clean["added_after_churn"].sum()) 

7774


In [83]:
# 'watched' flag vs. actual viewing history
pairs = watch[k].drop_duplicates()
chk = watchlist_clean.merge(pairs, on=k, how="left", indicator=True)
print(((chk["watched"]) & (chk["_merge"] == "left_only")).sum(), watchlist_clean["watched"].sum())
print(((~chk["watched"]) & (chk["_merge"] == "both")).sum())


29577 30020
485


In [84]:
print(watchlist_clean.shape)                                                  
watchlist_clean.to_csv("watchlist_clean.csv", index=False)
print(pd.read_csv("watchlist_clean.csv").shape)                               

(64939, 6)
(64939, 6)


Fixed:

# Converted added_date from text to datetime using the explicit format %Y-%m-%d. Nulls after conversion: 0.
# Removed 61 duplicate watchlist entries, the extra copies of 61 subscriber-title pairs. 31 of those pairs had conflicting watched values. Kept the most recent entry for each pair (ties on the same date broken by the higher watchlist_id), on the assumption that the latest entry reflects the current state. 65,000 rows became 64,939.


Validated (all passed clean):

# Row count matched the PRD (65,000 rows, 5 columns) before cleaning. No missing values. No duplicate watchlist_ids.
# Every watchlist entry links to a real subscriber and a real title (0 orphans), as the PRD asks.
# No entry was added before the subscriber's signup_date.
# No duplicate subscriber-title pairs remain after cleaning.


Added columns:

# added_after_churn: True for 7,774 entries (12.0%) added after the subscriber's churn_date. Kept and flagged.


Noted (not fixed):

# The watched flag does not agree with the viewing history. 29,577 of the 30,020 entries marked watched (98.5%) have no session for that subscriber and title in watch_history, and 485 entries marked not watched do have one. The flag was kept as defined and will be used as is for the watchlist conversion rate (30,020 of 64,939, 46.2%), but it cannot be verified against viewing data.
# 7,774 entries were added after the subscriber churned. As with the ratings, reviews and viewing sessions, the dates were not altered.


Result: watchlist table (65,000 rows) is cleaned and validated. 61 duplicate rows were removed, leaving 64,939. One column was added (added_after_churn), giving 6 columns. Exported as watchlist_clean.csv.

## Phase 1 Summary: Data Quality Report

The StreamFlix data is structurally sound: all six tables load with the expected row and column counts, there are no unexpected missing values, no duplicate primary keys, and no orphan subscriber or title IDs. The two columns with nulls (churn_date, 74.7%, and license_expiry, 21.8%) are legitimate, because they are blank exactly for active subscribers and for Originals. The real problems are logical, not structural. Dates are inconsistent across tables: sessions, ratings, reviews and watchlist entries are dated before signup or after churn, and ratings, reviews and the watched flag do not match the viewing history. Each case was flagged with a boolean column, not deleted, so that Phases 2 and 3 can run with and without the affected rows. The only rows removed were exact subscriber-title duplicates in ratings, reviews and watchlist, where the most recent record was kept.

| Table | Rows (raw to clean) | Main findings | Action |
|---|---|---|---|
| subscribers | 15,000 to 15,000 | 50 churn dates after data end (2026-05-31) | Flagged |
| titles | 9,000 to 9,000 | 1,285 popularity scores above the documented 0-1 range; 836 expired licences; 32 names shared by 85 titles | Flagged; none removed |
| watch_history | 650,000 to 650,000 | 131,199 sessions (20.2% of hours) after churn, from 3,708 subscribers | Flagged |
| ratings | 130,000 to 129,800 | 200 duplicate pairs (123 conflicting); 25,737 before signup; 21,330 after churn; 98.6% for unwatched titles | 200 removed; rest flagged |
| reviews | 110,000 to 109,829 | 171 duplicate pairs (81 conflicting); 22,062 before signup; 17,708 after churn; only 20 distinct texts | 171 removed; rest flagged |
| watchlist | 65,000 to 64,939 | 61 duplicate pairs (31 conflicting); 7,774 after churn; watched flag disagrees with history for 98.5% of watched entries | 61 removed; rest flagged |

Implication for later phases: KPIs based on viewing are run with and without the after_churn sessions, and ratings, reviews and the watchlist are not treated as evidence of what a subscriber watched.